# Pipeline Silver para Gold — CineData Analytics

Este notebook implementa a modelagem dimensional (*Star Schema*), o Data Mart de Contexto para Inteligência Artificial Generativa (*RAG*) e as consultas analíticas de negócio do projeto **CineData Analytics**.

### Entregas e Arquitetura:
1. **Tabelas Dimensão**: `dim_movies`, `dim_genres`, `dim_people`, `dim_companies`, `dim_reviews` com Chaves Substitutas (*Surrogate Keys*) `BIGINT`.
2. **Tabelas-Ponte (*Bridge Tables*)**: Resolução de relações *N:N* (`bridge_movie_genre`, `bridge_movie_person`, `bridge_movie_company`).
3. **Tabela Fato Central**: `fact_movies_performance` consolidando métricas financeiras e de engajamento por filme.
4. **Data Mart GenAI / RAG**: `gold_genai_movies_context` com documentos textuais enriquecidos e arquitetura anti-nulos (*NULL-safety*).
5. **Desafio de Analytics**: Resolução das 6 consultas estratégicas de negócio.

In [1]:
import warnings
from datetime import datetime
from zoneinfo import ZoneInfo

warnings.filterwarnings("ignore")

from pyspark.sql import Column, DataFrame, Row, SparkSession, Window
from pyspark.sql.functions import (
    add_months,
    array_join,
    avg,
    coalesce,
    col,
    collect_set,
    concat,
    concat_ws,
    count,
    current_date,
    format_number,
    lit,
    rank,
    row_number,
    slice,
    trim,
    when,
)
from pyspark.sql.functions import max as spark_max
from pyspark.sql.functions import round as spark_round
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.types import (
    DateType,
    DecimalType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# Obtenção da Sessão Spark gerenciada no Databricks Serverless
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.ansi.enabled", "false")

# Provisionamento do schema/database da camada Gold no catálogo gerenciado
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# Constantes globais da camada Gold
DECIMAL_FINANCIAL_PRECISION = "decimal(18,2)"
SURROGATE_KEY_DATA_TYPE = "bigint"
DECIMAL_AVERAGE_SCALE = 2
MAX_PRIMARY_ACTORS_COUNT = 5

def enforce_dataframe_schema(
    dataframe: DataFrame,
    expected_schema: StructType,
    strict_columns: bool = True
) -> DataFrame:
    actual_column_names = set(dataframe.columns)
    expected_column_names = [field.name for field in expected_schema.fields]
    missing_columns = set(expected_column_names) - actual_column_names

    if missing_columns:
        raise ValueError(f"Colunas obrigatórias ausentes no DataFrame: {missing_columns}")

    if strict_columns:
        unexpected_columns = actual_column_names - set(expected_column_names)
        if unexpected_columns:
            raise ValueError(f"Colunas imprevistas encontradas no DataFrame: {unexpected_columns}")

    ordered_column_expressions = [
        col(field.name).cast(field.dataType).alias(field.name)
        for field in expected_schema.fields
    ]
    return dataframe.select(ordered_column_expressions)

def write_dataframe(
    dataframe: DataFrame,
    table_name: str,
    save_mode: str = "overwrite"
) -> None:
    (
        dataframe.write
        .format("delta")
        .mode(save_mode)
        .saveAsTable(table_name)
    )

dq_execution_log: list[Row] = []

def execute_data_quality_check(
    table_name: str,
    check_name: str,
    target_dataframe: DataFrame,
    valid_condition: Column
) -> bool:
    total_records = target_dataframe.count()
    failed_records = target_dataframe.filter(~valid_condition).count()
    check_passed = (failed_records == 0)

    dq_execution_log.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_records=total_records,
            failed_records=failed_records,
            passed=check_passed,
            checked_at=datetime.now(ZoneInfo("America/Recife"))
        )
    )
    status_tag = "PASS" if check_passed else "FAIL"
    print(f"[{status_tag}] {table_name} | {check_name}: {failed_records}/{total_records} falhas.")
    return check_passed

def execute_uniqueness_check(
    table_name: str,
    check_name: str,
    target_dataframe: DataFrame,
    key_columns: list[str]
) -> bool:
    total_records = target_dataframe.count()
    duplicate_keys = (
        target_dataframe
        .groupBy(*key_columns)
        .count()
        .filter(col("count") > 1)
        .count()
    )
    check_passed = (duplicate_keys == 0)

    dq_execution_log.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_records=total_records,
            failed_records=duplicate_keys,
            passed=check_passed,
            checked_at=datetime.now(ZoneInfo("America/Recife"))
        )
    )
    status_tag = "PASS" if check_passed else "FAIL"
    print(f"[{status_tag}] {table_name} | {check_name}: {duplicate_keys} duplicatas em {total_records} registros.")
    return check_passed

def persist_data_quality_log() -> None:
    if dq_execution_log:
        dq_dataframe = spark.createDataFrame(dq_execution_log)
        write_dataframe(
            dataframe=dq_dataframe,
            table_name="gold.tb_data_quality_log",
            save_mode="append"
        )
        print("Log de Data Quality persistido com sucesso na tabela gold.tb_data_quality_log")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/20 14:03:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Construção da Dimensão Principal: `gold.dim_movies`

- **Origem**: `silver.tb_info_filmes`
- **Grão**: 1 registro único por filme (`id_filme`).
- **Chave Substituta**: `sk_movie_id` gerada deterministicamente por ordem de `id_filme`.

In [2]:
DimMoviesGoldSchema = StructType([
    StructField("sk_movie_id", LongType(), nullable=False),
    StructField("id_filme", StringType(), nullable=True),
    StructField("titulo", StringType(), nullable=True),
    StructField("data_lancamento", DateType(), nullable=True),
    StructField("ano_lancamento", IntegerType(), nullable=True),
    StructField("duracao_minutos", IntegerType(), nullable=True),
    StructField("idioma_original", StringType(), nullable=True),
    StructField("status_filme", StringType(), nullable=True),
    StructField("sinopse", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_dim_movies(raw_movies_dataframe: DataFrame) -> DataFrame:
    # Geração de chave substituta determinística (sk_movie_id) ordenada pela chave natural id_filme
    window_surrogate_key = Window.orderBy(col("id_filme").cast("integer").asc_nulls_last())

    # Armazenar os metadados principais e descritivos de cada filme em grão único por obra
    transformed_dataframe = (
        raw_movies_dataframe
        .withColumn("sk_movie_id", row_number().over(window_surrogate_key).cast(SURROGATE_KEY_DATA_TYPE))
        .select(
            col("sk_movie_id"),
            col("id_filme").cast("string"),
            col("titulo"),
            col("data_lancamento"),
            col("ano_lancamento"),
            col("duracao_minutos"),
            col("idioma_original"),
            col("status_filme"),
            col("sinopse"),
            col("ingestion_datetime")
        )
    )
    return enforce_dataframe_schema(transformed_dataframe, DimMoviesGoldSchema)

dataframe_movies_silver = spark.table("silver.tb_info_filmes")
dataframe_dim_movies = transform_dim_movies(dataframe_movies_silver)
write_dataframe(dataframe_dim_movies, table_name="gold.dim_movies")
dataframe_dim_movies.printSchema()
display(dataframe_dim_movies)

execute_uniqueness_check("gold.dim_movies", "unicidade_sk_movie_id", dataframe_dim_movies, ["sk_movie_id"])


root
 |-- sk_movie_id: long (nullable = false)
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



+-----------+--------+-----------------------------------+---------------+--------------+---------------+---------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+
|sk_movie_id|id_filme|titulo                             |data_lancamento|ano_lancamento|duracao_minutos|idioma_original|status_filme|sinopse                                                                                                                                                                                                                                                                                                                                       

[PASS] gold.dim_movies | unicidade_sk_movie_id: 0 duplicatas em 97879 registros.


True

## 2. Construção da Dimensão de Gêneros: `gold.dim_genres`

- **Origem**: `silver.tb_generos`
- **Grão**: 1 registro único por gênero canônico (`nome_genero`).
- **Chave Substituta**: `sk_genre_id` ordenada alfabeticamente.

In [3]:
DimGenresGoldSchema = StructType([
    StructField("sk_genre_id", LongType(), nullable=False),
    StructField("nome_genero", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_dim_genres(raw_genres_dataframe: DataFrame) -> DataFrame:
    # Catálogo único e deduplicado de todos os gêneros cinematográficos
    # Geração de chave substituta ordenada alfabeticamente
    window_surrogate_key = Window.orderBy(col("nome_genero"))

    distinct_genres = (
        raw_genres_dataframe
        .select("nome_genero", "ingestion_datetime")
        .filter(col("nome_genero").isNotNull())
        .dropDuplicates(["nome_genero"])
    )
    transformed_dataframe = (
        distinct_genres
        .withColumn("sk_genre_id", row_number().over(window_surrogate_key).cast(SURROGATE_KEY_DATA_TYPE))
        .select("sk_genre_id", "nome_genero", "ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, DimGenresGoldSchema)

dataframe_genres_silver = spark.table("silver.tb_generos")
dataframe_dim_genres = transform_dim_genres(dataframe_genres_silver)
write_dataframe(dataframe_dim_genres, table_name="gold.dim_genres")
dataframe_dim_genres.printSchema()
display(dataframe_dim_genres)

execute_uniqueness_check("gold.dim_genres", "unicidade_sk_genre_id", dataframe_dim_genres, ["sk_genre_id"])


root
 |-- sk_genre_id: long (nullable = false)
 |-- nome_genero: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



+-----------+-----------+--------------------------+
|sk_genre_id|nome_genero|ingestion_datetime        |
+-----------+-----------+--------------------------+
|1          |Action     |2026-09-19 23:55:34.748155|
|2          |Adventure  |2026-09-19 23:55:34.748155|
|3          |Animation  |2026-09-19 23:55:34.748155|
|4          |Comedy     |2026-09-19 23:55:34.748155|
|5          |Crime      |2026-09-19 23:55:34.748155|
|6          |Documentary|2026-09-19 23:55:34.748155|
|7          |Drama      |2026-09-19 23:55:34.748155|
|8          |Family     |2026-09-19 23:55:34.748155|
|9          |Fantasy    |2026-09-19 23:55:34.748155|
|10         |History    |2026-09-19 23:55:34.748155|
+-----------+-----------+--------------------------+
only showing top 10 rows


[PASS] gold.dim_genres | unicidade_sk_genre_id: 0 duplicatas em 19 registros.


True

## 3. Construção da Dimensão de Pessoas Físicas: `gold.dim_people`

- **Origem**: `silver.tb_pessoas_empresas`
- **Regra de Negócio**: Filtrar exclusivamente participantes pessoas físicas (`Ator`, `Diretor`, `Roteirista`).
- **Chave Substituta**: `sk_person_id` gerada deterministicamente.

In [4]:
DimPeopleGoldSchema = StructType([
    StructField("sk_person_id", LongType(), nullable=False),
    StructField("nome_pessoa", StringType(), nullable=True),
    StructField("tipo_pessoa", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_dim_people(raw_entities_dataframe: DataFrame) -> DataFrame:
    # Consolidar todas as pessoas físicas envolvidas na obra, filtrando exclusivamente
    # os tipos de entidade correspondentes a pessoas físicas: 'Ator', 'Diretor', 'Roteirista'.
    window_surrogate_key = Window.orderBy(col("tipo_pessoa"), col("nome_pessoa"))

    distinct_people = (
        raw_entities_dataframe
        .filter(col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
        .select(
            col("nome_entidade").alias("nome_pessoa"),
            col("tipo_entidade").alias("tipo_pessoa"),
            col("ingestion_datetime")
        )
        .dropDuplicates(["nome_pessoa", "tipo_pessoa"])
    )
    transformed_dataframe = (
        distinct_people
        .withColumn("sk_person_id", row_number().over(window_surrogate_key).cast(SURROGATE_KEY_DATA_TYPE))
        .select("sk_person_id", "nome_pessoa", "tipo_pessoa", "ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, DimPeopleGoldSchema)

dataframe_entities_silver = spark.table("silver.tb_pessoas_empresas")
dataframe_dim_people = transform_dim_people(dataframe_entities_silver)
write_dataframe(dataframe_dim_people, table_name="gold.dim_people")
dataframe_dim_people.printSchema()
display(dataframe_dim_people)

execute_uniqueness_check("gold.dim_people", "unicidade_sk_person_id", dataframe_dim_people, ["sk_person_id"])


root
 |-- sk_person_id: long (nullable = false)
 |-- nome_pessoa: string (nullable = true)
 |-- tipo_pessoa: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



+------------+----------------------------+-----------+--------------------------+
|sk_person_id|nome_pessoa                 |tipo_pessoa|ingestion_datetime        |
+------------+----------------------------+-----------+--------------------------+
|1           |1969 (\careful With That Axe|Ator       |2026-09-19 23:52:45.950925|
|2           |2 Chainz                    |Ator       |2026-09-19 23:52:45.950925|
|3           |2 Kupzz                     |Ator       |2026-09-19 23:52:45.950925|
|4           |2 Pretty Beebaby            |Ator       |2026-09-19 23:52:45.950925|
|5           |21 Savage                   |Ator       |2026-09-19 23:55:34.748155|
|6           |2am Ricky                   |Ator       |2026-09-19 23:55:34.748155|
|7           |3d                          |Ator       |2026-09-19 23:52:45.950925|
|8           |3d Natee                    |Ator       |2026-09-19 23:52:45.950925|
|9           |50 Cent                     |Ator       |2026-09-19 23:55:34.748155|
|10 

[PASS] gold.dim_people | unicidade_sk_person_id: 0 duplicatas em 420386 registros.


True

## 4. Construção da Dimensão de Produtoras / Estúdios: `gold.dim_companies`

- **Origem**: `silver.tb_pessoas_empresas`
- **Regra de Negócio**: Filtrar exclusivamente pessoas jurídicas (`tipo_entidade == "Produtora"`).
- **Chave Substituta**: `sk_company_id` gerada deterministicamente.

In [5]:
DimCompaniesGoldSchema = StructType([
    StructField("sk_company_id", LongType(), nullable=False),
    StructField("nome_produtora", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_dim_companies(raw_entities_dataframe: DataFrame) -> DataFrame:
    # Catálogo único de produtoras/estúdios, filtrando exclusivamente pessoas jurídicas ('Produtora')
    window_surrogate_key = Window.orderBy(col("nome_produtora"))

    distinct_companies = (
        raw_entities_dataframe
        .filter(col("tipo_entidade") == "Produtora")
        .select(
            col("nome_entidade").alias("nome_produtora"),
            col("ingestion_datetime")
        )
        .dropDuplicates(["nome_produtora"])
    )
    transformed_dataframe = (
        distinct_companies
        .withColumn("sk_company_id", row_number().over(window_surrogate_key).cast(SURROGATE_KEY_DATA_TYPE))
        .select("sk_company_id", "nome_produtora", "ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, DimCompaniesGoldSchema)

dataframe_dim_companies = transform_dim_companies(dataframe_entities_silver)
write_dataframe(dataframe_dim_companies, table_name="gold.dim_companies")
dataframe_dim_companies.printSchema()
display(dataframe_dim_companies)

execute_uniqueness_check("gold.dim_companies", "unicidade_sk_company_id", dataframe_dim_companies, ["sk_company_id"])


root
 |-- sk_company_id: long (nullable = false)
 |-- nome_produtora: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



+-------------+-----------------------------------+--------------------------+
|sk_company_id|nome_produtora                     |ingestion_datetime        |
+-------------+-----------------------------------+--------------------------+
|1            |#1nfluence Production              |2026-09-19 23:52:45.950925|
|2            |#beardforce Films                  |2026-09-19 23:52:45.950925|
|3            |#sinning Works                     |2026-09-19 23:55:34.748155|
|4            |& Extermination In An American City|2026-09-19 23:55:34.748155|
|5            |& Space Productions                |2026-09-19 23:55:34.748155|
|6            |((o))eco                           |2026-09-19 23:52:45.950925|
|7            |(mark Paul Wake                    |2026-09-19 23:55:34.748155|
|8            |(not) Heroine Movies               |2026-09-19 23:52:45.950925|
|9            |(notice Me) Kid Vicious            |2026-09-19 23:55:34.748155|
|10           |(pre)forma-se Artistic Productions |2

[PASS] gold.dim_companies | unicidade_sk_company_id: 0 duplicatas em 45353 registros.


True

## 5. Construção da Dimensão Resumida de Avaliações: `gold.dim_reviews`

- **Origem**: `silver.tb_avaliacoes_usuarios` combinada com `gold.dim_movies`
- **Regra de Negócio**: Agregação das notas por filme (`qtd_avaliacoes_usuarios` e `nota_media_usuarios`).
- **Chave Substituta**: `sk_review_id` gerada deterministicamente.

In [6]:
DimReviewsGoldSchema = StructType([
    StructField("sk_review_id", LongType(), nullable=False),
    StructField("sk_movie_id", LongType(), nullable=True),
    StructField("qtd_avaliacoes_usuarios", IntegerType(), nullable=True),
    StructField("nota_media_usuarios", DoubleType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_dim_reviews(raw_reviews_dataframe: DataFrame, canonical_movies_dataframe: DataFrame) -> DataFrame:
    # Consolidar as avaliações individuais geradas pelos usuários, transformando-as em uma métrica
    # resumida por filme: contagem de avaliações e média arredondada em 2 casas decimais.
    window_surrogate_key = Window.orderBy(col("sk_movie_id"))

    aggregated_reviews = (
        raw_reviews_dataframe
        .groupBy("id_filme")
        .agg(
            count("nota_usuario").cast("integer").alias("qtd_avaliacoes_usuarios"),
            spark_round(avg("nota_usuario"), DECIMAL_AVERAGE_SCALE).alias("nota_media_usuarios"),
            spark_max("ingestion_datetime").alias("ingestion_datetime")
        )
    )

    # Conexão com dim_movies para resolução da chave estrangeira sk_movie_id
    transformed_dataframe = (
        aggregated_reviews
        .join(
            canonical_movies_dataframe.select("id_filme", "sk_movie_id"),
            canonical_movies_dataframe["id_filme"] == aggregated_reviews["id_filme"].cast("string"),
            "inner"
        )
        .withColumn("sk_review_id", row_number().over(window_surrogate_key).cast(SURROGATE_KEY_DATA_TYPE))
        .select(
            col("sk_review_id"),
            col("sk_movie_id"),
            col("qtd_avaliacoes_usuarios"),
            col("nota_media_usuarios"),
            aggregated_reviews["ingestion_datetime"]
        )
    )
    return enforce_dataframe_schema(transformed_dataframe, DimReviewsGoldSchema)

dataframe_reviews_silver = spark.table("silver.tb_avaliacoes_usuarios")
dataframe_dim_reviews = transform_dim_reviews(dataframe_reviews_silver, dataframe_dim_movies)
write_dataframe(dataframe_dim_reviews, table_name="gold.dim_reviews")
dataframe_dim_reviews.printSchema()
display(dataframe_dim_reviews)

execute_uniqueness_check("gold.dim_reviews", "unicidade_sk_review_id", dataframe_dim_reviews, ["sk_review_id"])


root
 |-- sk_review_id: long (nullable = false)
 |-- sk_movie_id: long (nullable = false)
 |-- qtd_avaliacoes_usuarios: integer (nullable = false)
 |-- nota_media_usuarios: double (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



+------------+-----------+-----------------------+-------------------+--------------------------+
|sk_review_id|sk_movie_id|qtd_avaliacoes_usuarios|nota_media_usuarios|ingestion_datetime        |
+------------+-----------+-----------------------+-------------------+--------------------------+
|1           |2          |1                      |8.3                |2026-09-19 23:55:35.299475|
|2           |3          |2                      |5.15               |2026-09-19 23:55:35.299475|
|3           |4          |1                      |2.5                |2026-09-19 23:55:35.299475|
|4           |5          |1                      |0.1                |2026-09-19 23:55:35.299475|
|5           |10         |1                      |0.2                |2026-09-19 23:55:35.299475|
+------------+-----------+-----------------------+-------------------+--------------------------+
only showing top 5 rows


[PASS] gold.dim_reviews | unicidade_sk_review_id: 0 duplicatas em 27303 registros.


True

## 6. Construção das Tabelas-Ponte (*Bridge Tables*)

Resolução das relações *N:N* de filmes com gêneros, pessoas e produtoras:
- `gold.bridge_movie_genre`
- `gold.bridge_movie_person`
- `gold.bridge_movie_company`

In [7]:
BridgeMovieGenreGoldSchema = StructType([
    StructField("sk_movie_id", LongType(), nullable=False),
    StructField("sk_genre_id", LongType(), nullable=False),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

BridgeMoviePersonGoldSchema = StructType([
    StructField("sk_movie_id", LongType(), nullable=False),
    StructField("sk_person_id", LongType(), nullable=False),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

BridgeMovieCompanyGoldSchema = StructType([
    StructField("sk_movie_id", LongType(), nullable=False),
    StructField("sk_company_id", LongType(), nullable=False),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

# Como um filme pode ter vários gêneros, vários atores e várias produtoras (e vice-versa),
# o uso de tabelas-ponte (bridge tables) é estritamente necessário para conectar a tabela dim_movies
# às dimensões periféricas sem duplicar o registro da tabela Fato.
def build_bridge_movie_genre(
    canonical_genres_dataframe: DataFrame,
    dim_movies_dataframe: DataFrame,
    dim_genres_dataframe: DataFrame
) -> DataFrame:
    transformed_dataframe = (
        canonical_genres_dataframe
        .join(
            dim_movies_dataframe.select("id_filme", "sk_movie_id"),
            dim_movies_dataframe["id_filme"] == canonical_genres_dataframe["id_filme"].cast("string"),
            "inner"
        )
        .join(dim_genres_dataframe.select("nome_genero", "sk_genre_id"), "nome_genero", "inner")
        .select("sk_movie_id", "sk_genre_id", canonical_genres_dataframe["ingestion_datetime"])
        .dropDuplicates(["sk_movie_id", "sk_genre_id"])
    )
    return enforce_dataframe_schema(transformed_dataframe, BridgeMovieGenreGoldSchema)

def build_bridge_movie_person(
    canonical_entities_dataframe: DataFrame,
    dim_movies_dataframe: DataFrame,
    dim_people_dataframe: DataFrame
) -> DataFrame:
    people_entities = canonical_entities_dataframe.filter(col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
    transformed_dataframe = (
        people_entities
        .join(
            dim_movies_dataframe.select("id_filme", "sk_movie_id"),
            dim_movies_dataframe["id_filme"] == people_entities["id_filme"].cast("string"),
            "inner"
        )
        .join(
            dim_people_dataframe.select("nome_pessoa", "tipo_pessoa", "sk_person_id"),
            (dim_people_dataframe["nome_pessoa"] == people_entities["nome_entidade"]) &
            (dim_people_dataframe["tipo_pessoa"] == people_entities["tipo_entidade"]),
            "inner"
        )
        .select("sk_movie_id", "sk_person_id", people_entities["ingestion_datetime"])
        .dropDuplicates(["sk_movie_id", "sk_person_id"])
    )
    return enforce_dataframe_schema(transformed_dataframe, BridgeMoviePersonGoldSchema)

def build_bridge_movie_company(
    canonical_entities_dataframe: DataFrame,
    dim_movies_dataframe: DataFrame,
    dim_companies_dataframe: DataFrame
) -> DataFrame:
    company_entities = canonical_entities_dataframe.filter(col("tipo_entidade") == "Produtora")
    transformed_dataframe = (
        company_entities
        .join(
            dim_movies_dataframe.select("id_filme", "sk_movie_id"),
            dim_movies_dataframe["id_filme"] == company_entities["id_filme"].cast("string"),
            "inner"
        )
        .join(
            dim_companies_dataframe.select("nome_produtora", "sk_company_id"),
            dim_companies_dataframe["nome_produtora"] == company_entities["nome_entidade"],
            "inner"
        )
        .select("sk_movie_id", "sk_company_id", company_entities["ingestion_datetime"])
        .dropDuplicates(["sk_movie_id", "sk_company_id"])
    )
    return enforce_dataframe_schema(transformed_dataframe, BridgeMovieCompanyGoldSchema)

dataframe_bridge_movie_genre = build_bridge_movie_genre(dataframe_genres_silver, dataframe_dim_movies, dataframe_dim_genres)
write_dataframe(dataframe_bridge_movie_genre, table_name="gold.bridge_movie_genre")

dataframe_bridge_movie_person = build_bridge_movie_person(dataframe_entities_silver, dataframe_dim_movies, dataframe_dim_people)
write_dataframe(dataframe_bridge_movie_person, table_name="gold.bridge_movie_person")

dataframe_bridge_movie_company = build_bridge_movie_company(dataframe_entities_silver, dataframe_dim_movies, dataframe_dim_companies)
write_dataframe(dataframe_bridge_movie_company, table_name="gold.bridge_movie_company")

print("Tabelas-Ponte geradas com sucesso.")


Tabelas-Ponte geradas com sucesso.


## 7. Construção da Tabela Fato Central: `gold.fact_movies_performance`

- **Objetivo**: Centralizar todas as métricas contábeis (orçamento, receita, lucro em USD e BRL) e métricas de engajamento.
- **Grão**: 1 registro único por filme (`sk_movie_id`).

In [8]:
FactMoviesPerformanceGoldSchema = StructType([
    StructField("sk_movie_id", LongType(), nullable=False),
    StructField("orcamento_usd", DecimalType(18, 2), nullable=True),
    StructField("receita_usd", DecimalType(18, 2), nullable=True),
    StructField("lucro_usd", DecimalType(18, 2), nullable=True),
    StructField("orcamento_brl", DecimalType(18, 2), nullable=True),
    StructField("receita_brl", DecimalType(18, 2), nullable=True),
    StructField("lucro_brl", DecimalType(18, 2), nullable=True),
    StructField("popularidade", DoubleType(), nullable=True),
    StructField("nota_media_tmdb", DoubleType(), nullable=True),
    StructField("qtd_votos_tmdb", IntegerType(), nullable=True),
    StructField("nota_media_imdb", DoubleType(), nullable=True),
    StructField("qtd_votos_imdb", IntegerType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_fact_movies_performance(
    canonical_movies_dataframe: DataFrame,
    financial_metrics_dataframe: DataFrame,
    engagement_metrics_dataframe: DataFrame
) -> DataFrame:
    # Grão: Um registro único por filme.
    # Objetivo: Centralizar todas as métricas financeiras e de engajamento do filme.
    # A tabela fato deve consolidar métricas sem duplicar o grão através dos joins (LEFT JOIN a partir de dim_movies).
    joined_df = (
        canonical_movies_dataframe.select("id_filme", "sk_movie_id")
        .join(
            financial_metrics_dataframe,
            financial_metrics_dataframe["id_filme"].cast("string") == canonical_movies_dataframe["id_filme"],
            "left"
        )
        .join(
            engagement_metrics_dataframe,
            engagement_metrics_dataframe["id_filme"].cast("string") == canonical_movies_dataframe["id_filme"],
            "left"
        )
    )

    transformed_dataframe = joined_df.select(
        col("sk_movie_id"),
        financial_metrics_dataframe["orcamento_usd"],
        financial_metrics_dataframe["receita_usd"],
        financial_metrics_dataframe["lucro_usd"],
        financial_metrics_dataframe["orcamento_brl"],
        financial_metrics_dataframe["receita_brl"],
        financial_metrics_dataframe["lucro_brl"],
        engagement_metrics_dataframe["popularidade"],
        engagement_metrics_dataframe["nota_media_tmdb"],
        engagement_metrics_dataframe["qtd_votos_tmdb"],
        engagement_metrics_dataframe["nota_media_imdb"],
        engagement_metrics_dataframe["qtd_votos_imdb"],
        coalesce(financial_metrics_dataframe["ingestion_datetime"], engagement_metrics_dataframe["ingestion_datetime"]).alias("ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, FactMoviesPerformanceGoldSchema)

dataframe_financials_silver = spark.table("silver.tb_financeiro_filmes")
dataframe_metrics_silver = spark.table("silver.tb_metricas_engajamento")
dataframe_fact_movies_performance = transform_fact_movies_performance(
    canonical_movies_dataframe=dataframe_dim_movies,
    financial_metrics_dataframe=dataframe_financials_silver,
    engagement_metrics_dataframe=dataframe_metrics_silver
)
write_dataframe(dataframe_fact_movies_performance, table_name="gold.fact_movies_performance")
dataframe_fact_movies_performance.printSchema()
display(dataframe_fact_movies_performance)

execute_uniqueness_check("gold.fact_movies_performance", "unicidade_sk_movie_id", dataframe_fact_movies_performance, ["sk_movie_id"])
execute_data_quality_check("gold.fact_movies_performance", "contagem_registros_valida", dataframe_fact_movies_performance, col("sk_movie_id").isNotNull())


root
 |-- sk_movie_id: long (nullable = false)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



+-----------+-------------+------------+------------+-------------+-------------+-------------+------------+---------------+--------------+---------------+--------------+--------------------------+
|sk_movie_id|orcamento_usd|receita_usd |lucro_usd   |orcamento_brl|receita_brl  |lucro_brl    |popularidade|nota_media_tmdb|qtd_votos_tmdb|nota_media_imdb|qtd_votos_imdb|ingestion_datetime        |
+-----------+-------------+------------+------------+-------------+-------------+-------------+------------+---------------+--------------+---------------+--------------+--------------------------+
|1          |25000000.00  |83080890.00 |58080890.00 |129000000.00 |428697392.40 |299697392.40 |24.584      |4.966          |2375          |NULL           |46286         |2026-09-19 23:55:33.697557|
|2          |NULL         |NULL        |NULL        |NULL         |NULL         |NULL         |8.929       |7.064          |118           |6.6            |4617          |2026-09-19 23:55:33.697557|
|3        

[PASS] gold.fact_movies_performance | unicidade_sk_movie_id: 0 duplicatas em 97879 registros.


[PASS] gold.fact_movies_performance | contagem_registros_valida: 0/97879 falhas.


True

## 8. Construção do Data Mart GenAI / RAG: `gold.gold_genai_movies_context`

- **Objetivo**: Fornecer documentos textuais contextualizados para alimentação de base vetorial (*Vector Search*).
- **Arquitetura Anti-Nulos (*NULL-Safety*)**: Fallbacks descritivos para evitar perda de documentos em concatenações.

In [9]:
GoldGenaiMoviesContextSchema = StructType([
    StructField("movie_id", StringType(), nullable=True),
    StructField("title", StringType(), nullable=True),
    StructField("llm_context_document", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_genai_movies_context(
    canonical_movies_dataframe: DataFrame,
    performance_fact_dataframe: DataFrame,
    bridge_person_dataframe: DataFrame,
    canonical_people_dataframe: DataFrame
) -> DataFrame:
    # Agregação de múltiplos atores e diretores por filme via bridge_movie_person
    people_with_role = bridge_person_dataframe.join(canonical_people_dataframe, "sk_person_id", "inner")

    actors_aggregated = (
        people_with_role
        .filter(col("tipo_pessoa") == "Ator")
        .groupBy("sk_movie_id")
        .agg(array_join(slice(collect_set("nome_pessoa"), 1, MAX_PRIMARY_ACTORS_COUNT), ", ").alias("atores_principais"))
    )
    directors_aggregated = (
        people_with_role
        .filter(col("tipo_pessoa") == "Diretor")
        .groupBy("sk_movie_id")
        .agg(concat_ws(", ", collect_set("nome_pessoa")).alias("diretores"))
    )

    joined_df = (
        canonical_movies_dataframe.select("sk_movie_id", "id_filme", "titulo", "ano_lancamento", "sinopse", "ingestion_datetime")
        .join(performance_fact_dataframe.select("sk_movie_id", "receita_usd", "orcamento_usd"), "sk_movie_id", "left")
        .join(actors_aggregated, "sk_movie_id", "left")
        .join(directors_aggregated, "sk_movie_id", "left")
    )

    # Atenção Técnica — A 'casca de banana' dos nulos:
    # Funções de concatenação (concat(), operador ||) retornam NULL para a string inteira se qualquer campo for nulo.
    # Um único campo director ou overview vazio pode fazer o filme inteiro desaparecer silenciosamente da tabela de contexto.
    # Portanto, aplica-se fallback descritivo e seguro para cada campo antes da concatenação final.
    texto_titulo = coalesce(trim(col("titulo")), lit("Título não informado"))
    texto_ano = when(col("ano_lancamento").isNotNull(), col("ano_lancamento").cast("string")).otherwise(lit("ano não informado"))
    texto_receita = when(col("receita_usd").isNotNull() & (col("receita_usd") > 0), concat(lit("US$ "), format_number(col("receita_usd"), DECIMAL_AVERAGE_SCALE))).otherwise(lit("valor não informado"))
    texto_orcamento = when(col("orcamento_usd").isNotNull() & (col("orcamento_usd") > 0), concat(lit("US$ "), format_number(col("orcamento_usd"), DECIMAL_AVERAGE_SCALE))).otherwise(lit("valor não informado"))
    texto_atores = when(col("atores_principais").isNotNull() & (trim(col("atores_principais")) != ""), trim(col("atores_principais"))).otherwise(lit("elenco não informado"))
    texto_diretores = when(col("diretores").isNotNull() & (trim(col("diretores")) != ""), trim(col("diretores"))).otherwise(lit("diretor não informado"))
    texto_sinopse = coalesce(trim(col("sinopse")), lit("Sinopse não disponível."))

    # Template esperado da coluna llm_context_document:
    # 'O filme [TÍTULO], lançado no ano de [ANO], faturou [RECEITA] e teve um custo de [ORÇAMENTO].
    # Estrelado por [ATORES PRINCIPAIS] e dirigido por [DIRETOR], o filme possui a seguinte sinopse: [OVERVIEW].'
    context_doc_expr = concat(
        lit("O filme "), texto_titulo,
        lit(", lançado no ano de "), texto_ano,
        lit(", faturou "), texto_receita,
        lit(" e teve um custo de "), texto_orcamento,
        lit(". Estrelado por "), texto_atores,
        lit(" e dirigido por "), texto_diretores,
        lit(", o filme possui a seguinte sinopse: "), texto_sinopse
    )

    transformed_dataframe = joined_df.select(
        col("id_filme").alias("movie_id"),
        col("titulo").alias("title"),
        context_doc_expr.alias("llm_context_document"),
        col("ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, GoldGenaiMoviesContextSchema)

dataframe_gold_genai_movies_context = transform_genai_movies_context(
    canonical_movies_dataframe=dataframe_dim_movies,
    performance_fact_dataframe=dataframe_fact_movies_performance,
    bridge_person_dataframe=dataframe_bridge_movie_person,
    canonical_people_dataframe=dataframe_dim_people
)
write_dataframe(dataframe_gold_genai_movies_context, table_name="gold.gold_genai_movies_context")
dataframe_gold_genai_movies_context.printSchema()
display(dataframe_gold_genai_movies_context)


root
 |-- movie_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- llm_context_document: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



+--------+-------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+
|movie_id|title              |llm_context_document                                                                                                       

## 9. Desafio de Analytics (Consultas Estratégicas de Negócio)

Execução das 6 consultas analíticas estratégicas sobre o modelo dimensional.

In [10]:
TOP_POPULARITY_LIMIT = 5
TOP_REVENUE_RANK_LIMIT = 10
TOP_ACTORS_LIMIT = 5
TOP_COMPANIES_LIMIT = 5
TWO_YEARS_MONTHS_OFFSET = -24
FIVE_YEARS_MONTHS_OFFSET = -60

# 1. Qual é a receita total (em R$) somada de todos os filmes da base?
query_1_receita_total_brl = dataframe_fact_movies_performance.select(
    spark_sum(col("receita_brl")).cast(DECIMAL_FINANCIAL_PRECISION).alias("receita_total_brl")
)
display(query_1_receita_total_brl)

# 2. Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade.
query_2_top_5_popularidade = (
    dataframe_fact_movies_performance
    .join(dataframe_dim_movies, "sk_movie_id", "inner")
    .select("titulo", "popularidade")
    .orderBy(col("popularidade").desc_nulls_last())
    .limit(TOP_POPULARITY_LIMIT)
)
display(query_2_top_5_popularidade)

# 3. Quantos filmes cada gênero possui? Liste do maior para o menor volume.
query_3_filmes_por_genero = (
    dataframe_bridge_movie_genre
    .join(dataframe_dim_genres, "sk_genre_id", "inner")
    .groupBy("nome_genero")
    .agg(count("sk_movie_id").alias("quantidade_filmes"))
    .orderBy(col("quantidade_filmes").desc())
)
display(query_3_filmes_por_genero)

# 4. Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking (RANK()).
window_rank_revenue = Window.orderBy(col("receita_usd").desc_nulls_last())
query_4_top_10_receita = (
    dataframe_fact_movies_performance
    .join(dataframe_dim_movies, "sk_movie_id", "inner")
    .filter(col("receita_usd").isNotNull())
    .withColumn("posicao_ranking", rank().over(window_rank_revenue))
    .select("posicao_ranking", "titulo", "receita_usd", "receita_brl")
    .orderBy("posicao_ranking")
    .limit(TOP_REVENUE_RANK_LIMIT)
)
display(query_4_top_10_receita)

# Regra de negócio temporal: Para o recorte dos últimos 2 e 5 anos, considere como data limite superior
# a data de lançamento realizada mais recente na base (ignorando registros com datas futuras ou não lançadas).
max_release_date_record = (
    dataframe_dim_movies
    .filter(col("data_lancamento").isNotNull() & (col("status_filme") == "Lançado") & (col("data_lancamento") <= current_date()))
    .select(spark_max("data_lancamento").alias("max_data_lancamento"))
    .collect()
)
reference_max_release_date = max_release_date_record[0]["max_data_lancamento"]
date_window_two_years = dataframe_dim_movies.select(add_months(lit(reference_max_release_date), TWO_YEARS_MONTHS_OFFSET).alias("limite_data")).collect()[0]["limite_data"]
date_window_five_years = dataframe_dim_movies.select(add_months(lit(reference_max_release_date), FIVE_YEARS_MONTHS_OFFSET).alias("limite_data")).collect()[0]["limite_data"]

# 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos?
query_5_top_atores_2_anos = (
    dataframe_dim_movies
    .filter(col("data_lancamento").between(date_window_two_years, reference_max_release_date) & (col("status_filme") == "Lançado"))
    .join(dataframe_bridge_movie_person, "sk_movie_id", "inner")
    .join(dataframe_dim_people.filter(col("tipo_pessoa") == "Ator"), "sk_person_id", "inner")
    .groupBy("nome_pessoa")
    .agg(count(dataframe_bridge_movie_person["sk_movie_id"]).alias("quantidade_participacoes"))
    .orderBy(col("quantidade_participacoes").desc())
    .limit(TOP_ACTORS_LIMIT)
)
display(query_5_top_atores_2_anos)

# 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos?
query_6_top_produtoras_5_anos = (
    dataframe_dim_movies
    .filter(col("data_lancamento").between(date_window_five_years, reference_max_release_date) & (col("status_filme") == "Lançado"))
    .join(dataframe_fact_movies_performance, "sk_movie_id", "inner")
    .join(dataframe_bridge_movie_company, "sk_movie_id", "inner")
    .join(dataframe_dim_companies, "sk_company_id", "inner")
    .groupBy("nome_produtora")
    .agg(spark_sum("lucro_usd").cast(DECIMAL_FINANCIAL_PRECISION).alias("lucro_total_usd"))
    .orderBy(col("lucro_total_usd").desc_nulls_last())
    .limit(TOP_COMPANIES_LIMIT)
)
display(query_6_top_produtoras_5_anos)


+-----------------+
|receita_total_brl|
+-----------------+
|838275201039.60  |
+-----------------+



+-------------------------------------+------------+
|titulo                               |popularidade|
+-------------------------------------+------------+
|blue beetle                          |2994.357    |
|Gran Turismo                         |2680.593    |
|La Fellinette                        |2020.0      |
|The Fear Footage 2: Curse of the Tape|2019.0      |
|wwe survivor series 2018             |2018.0      |
+-------------------------------------+------------+



+---------------+-----------------+
|nome_genero    |quantidade_filmes|
+---------------+-----------------+
|Drama          |32286            |
|Documentary    |18996            |
|Comedy         |18624            |
|Thriller       |10274            |
|Horror         |9729             |
|Romance        |7639             |
|Action         |6049             |
|Crime          |4747             |
|Animation      |4469             |
|TV Movie       |4079             |
|Science Fiction|3769             |
|Family         |3722             |
|Mystery        |3317             |
|Fantasy        |3278             |
|Adventure      |2870             |
|Music          |2792             |
|History        |2416             |
|War            |960              |
|Western        |410              |
+---------------+-----------------+



+---------------+---------------------------+-------------+--------------+
|posicao_ranking|titulo                     |receita_usd  |receita_brl   |
+---------------+---------------------------+-------------+--------------+
|1              |Avengers: Endgame          |2800000000.00|14448000000.00|
|2              |Avatar: The Way of Water   |2320250281.00|11972491449.96|
|3              |AVENGERS: INFINITY WAR     |2052415039.00|10590461601.24|
|4              |spider-man: no way home    |1921847111.00|9916731092.76 |
|5              |The Lion King              |1663075401.00|8581469069.16 |
|6              |Top Gun: Maverick          |1488732821.00|7681861356.36 |
|7              |Barbie                     |1428545028.00|7371292344.48 |
|8              |The Super Mario Bros. Movie|1355725263.00|6995542357.08 |
|9              |Black Panther              |1349926083.00|6965618588.28 |
|10             |Star Wars: The Last Jedi   |1332698830.00|6876725962.80 |
+---------------+--------

+--------------+------------------------+
|nome_pessoa   |quantidade_participacoes|
+--------------+------------------------+
|Kevin Hart    |64                      |
|Melissa Ponzio|59                      |
|Josh Hartnett |59                      |
|John Travolta |59                      |
|John Cena     |59                      |
+--------------+------------------------+



+------------------+---------------+
|nome_produtora    |lucro_total_usd|
+------------------+---------------+
|Universal Pictures|5772329679.00  |
|Marvel Studios    |4953462823.00  |
|Columbia Pictures |3662050755.00  |
|Pascal Pictures   |2701952454.00  |
|Illumination      |2431353473.00  |
+------------------+---------------+



## 10. Consolidação e Persistência do Log de Data Quality (Camada Gold)

Persistência e auditoria final das métricas de integridade dimensional da camada Gold.

In [11]:
persist_data_quality_log()
dataframe_dq_gold_log = spark.table("gold.tb_data_quality_log")
display(dataframe_dq_gold_log.orderBy(col("checked_at").desc()))


Log de Data Quality persistido com sucesso em: /home/miguelsb/workspace/visagio/data/gold/tb_data_quality_log
+----------------------------+-------------------------+-------------+--------------+------+--------------------------+
|table_name                  |check_name               |total_records|failed_records|passed|checked_at                |
+----------------------------+-------------------------+-------------+--------------+------+--------------------------+
|gold.fact_movies_performance|contagem_registros_valida|97879        |0             |true  |2026-09-20 14:03:55.2173  |
|gold.fact_movies_performance|unicidade_sk_movie_id    |97879        |0             |true  |2026-09-20 14:03:55.005436|
|gold.dim_reviews            |unicidade_sk_review_id   |27303        |0             |true  |2026-09-20 14:03:48.80097 |
|gold.dim_companies          |unicidade_sk_company_id  |45353        |0             |true  |2026-09-20 14:03:46.906806|
|gold.dim_people             |unicidade_sk_person_